# Lab 2: The Contract of a Tool — Pydantic Schemas

**Series**: Agentic AI Science Playbook — Foundation Layer
**Goal**: Use Pydantic + OpenAI function calling so the LLM returns typed, validated arguments.
**Time**: ~45 min.

## Problem → Solution

| Before | After |
|--------|-------|
| `TOOL: search_literature` | `{query, max_results, year_from, domain}` |
| Raw user string as arg | Typed Pydantic object |
| Regex breaks on drift | OpenAI function calling is robust |

## Learning Objectives

By the end of this lab, you will be able to:
- Define tool interfaces using Pydantic `BaseModel` schemas
- Convert Pydantic schemas to OpenAI function calling format
- Implement type-safe tool execution with validated arguments
- Understand why schema-driven agents are more reliable than regex-based parsing

## Why This Matters for Scientists

Scientific tools have **complex parameters**: a molecular docking tool needs a protein structure, a ligand SMILES string, a scoring function, and a pose count. A climate model needs coordinates, time range, resolution, and variables.

Without schemas, the LLM guesses at parameter formats and your code breaks silently. With Pydantic schemas, every parameter is **documented, typed, and validated** — the LLM knows exactly what each tool expects, and invalid inputs are caught before execution.

> **Real-world example**: In drug discovery pipelines, a malformed SMILES string passed to a docking tool wastes hours of GPU compute. Pydantic schemas prevent this at the agent level — before the tool ever runs.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Lab 0, Lab 1 |
| Concepts | Tool selection, system prompts |
| New library | `pydantic` (v2) |

**NVIDIA Connection**: NVIDIA NIM endpoints support **OpenAI-compatible function calling**. The Pydantic → JSON schema → tools format pipeline you build here works identically with `nvidia/llama-3.3-nemotron-super-49b-v1.5` on NIM. Nemotron models are specifically optimized for structured output and function calling.

### Install Dependencies
We add `pydantic` alongside `openai`. Pydantic is the industry-standard library for data validation in Python.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Same dual-provider setup — works with both OpenAI and NVIDIA NIM.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field

## Concept: From Strings to Schemas

In Lab 0, tools were just description strings:
```python
TOOLS = {"search_literature": "Search scientific databases for papers..."}
```

The LLM responded with `TOOL: search_literature` and we passed the raw user message as the argument. This has three problems:

| Problem | Example | Consequence |
|---------|---------|-------------|
| **No type safety** | `max_results="five"` instead of `5` | Runtime error in tool |
| **No defaults** | User doesn't specify max_results | Tool crashes or returns too many |
| **No documentation** | LLM doesn't know what args are available | Guesses wrong parameters |

**Pydantic schemas solve all three** by defining a formal contract between the LLM and your tools:

```
Pydantic BaseModel → JSON Schema → OpenAI tools format → LLM fills in typed args → Pydantic validates
```

## Step 1: Define Tool Schemas

### Define the Search Tool Schema
Instead of a plain description string, we define a **Pydantic model** with typed fields. Each field has a type (`str`, `int`, `None`), a default value, and a description. The LLM reads these descriptions to know what arguments to provide.

In [ ]:
class SearchLiteratureArgs(BaseModel):
    query: str = Field(..., description="Search query — keywords or a sentence.")
    max_results: int = Field(5, description="Max papers to return (1-20).")
    year_from: int | None = Field(None, description="Filter to papers from this year onward.")
    domain: str | None = Field(None, description="Scientific domain e.g. 'genomics', 'cardiology'.")

### Define the Summarize Tool Schema
Same pattern for our second tool. Let's also print the auto-generated JSON schema — this is exactly what the LLM sees.

In [ ]:
class SummarizeFindingsArgs(BaseModel):
    text: str = Field(..., description="Content to summarize.")
    style: str = Field("bullet_points", description="bullet_points | paragraph | table.")
    max_sentences: int = Field(5, description="Max sentences or bullets.")

print("SearchLiteratureArgs schema:")
print(json.dumps(SearchLiteratureArgs.model_json_schema(), indent=2))

## Step 2: Convert to OpenAI Tool Format

### Convert Schemas to OpenAI Tool Format
The `pydantic_to_openai_tool()` helper converts our Pydantic models into the JSON format that OpenAI/NIM function calling expects. This is a one-time conversion — you define the schema once and the format is generated automatically.

In [ ]:
def pydantic_to_openai_tool(name, description, cls):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": cls.model_json_schema()}}

OPENAI_TOOLS = [
    pydantic_to_openai_tool("search_literature", "Search scientific databases for papers.", SearchLiteratureArgs),
    pydantic_to_openai_tool("summarize_findings", "Summarize research results.", SummarizeFindingsArgs),
]
SCHEMA_MAP = {"search_literature": SearchLiteratureArgs, "summarize_findings": SummarizeFindingsArgs}
print("Tools:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Concept: The Function Calling Protocol

OpenAI function calling (also supported by NVIDIA NIM) replaces our fragile regex parsing with a structured protocol:

1. **You define tools** as JSON schemas (auto-generated from Pydantic)
2. **The LLM returns** a structured `tool_calls` object (not free text)
3. **You parse** typed arguments from the JSON response
4. **Pydantic validates** the arguments before execution

This is a massive reliability upgrade:

| Aspect | Lab 0 (regex) | Lab 2 (function calling) |
|--------|--------------|--------------------------|
| Parsing | Fragile regex | Structured JSON |
| Arguments | Raw string | Typed, validated |
| Defaults | Not possible | Automatic |
| Error handling | Silent failures | Clear validation errors |

> **This is the pattern used in production.** Every serious agent framework (LangChain, LlamaIndex, CrewAI, NeMo Agent Toolkit) uses function calling with schemas.

## Step 3: Agent with Function Calling

### Build the Agent with Function Calling
This version uses OpenAI's `tools` parameter instead of our manual regex parsing. The LLM returns structured `tool_calls` with typed arguments — no more guessing. Notice `tool_choice='auto'` lets the model decide whether to call a tool.

In [ ]:
def call_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[
            {"role": "system", "content": "You are a scientific research assistant. Use the provided tools."},
            {"role": "user", "content": user_message},
        ],
        tools=OPENAI_TOOLS, tool_choice="auto",
    )
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "args": None}
    call = msg.tool_calls[0]
    raw = json.loads(call.function.arguments)
    validated = SCHEMA_MAP[call.function.name](**raw) if call.function.name in SCHEMA_MAP else raw
    return {"tool": call.function.name, "args": validated}

for q in [
    "Find 10 cited CRISPR papers after 2020 in genomics.",
    "Summarize in 4 bullet points: mRNA vaccine advances.",
    "Search 3 ocean acidification papers from 2022.",
]:
    r = call_agent(q)
    print(f"Q: {q[:65]}")
    print(f"  Tool: {r['tool']}")
    print(f"  Args: {r['args']}")
    print()

<details>
<summary>Expected output (click to expand)</summary>

```
Q: Find 10 cited CRISPR papers after 2020 in genomics.
  Tool: search_literature
  Args: query='CRISPR' max_results=10 year_from=2020 domain='genomics'

Q: Summarize in 4 bullet points: mRNA vaccine advances.
  Tool: summarize_findings
  Args: text='mRNA vaccine advances' style='bullet_points' max_sentences=4

Q: Search 3 ocean acidification papers from 2022.
  Tool: search_literature
  Args: query='ocean acidification' max_results=3 year_from=2022 domain=None
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Step 4: Execute with Typed Arguments

### Implement Typed Tool Execution
Now our tool functions receive **validated Pydantic objects** instead of raw strings. Notice how `args.query`, `args.max_results`, etc. are already typed and validated — no parsing needed.

In [ ]:
def execute_search(args: SearchLiteratureArgs) -> str:
    d = f" in {args.domain}" if args.domain else ""
    y = f" from {args.year_from}+" if args.year_from else ""
    return (
        f"[search_literature] '{args.query}'{d}{y}\n"
        f"  Returning up to {args.max_results} results (stub):\n"
        "  1. Paper A — Smith et al. (2023)\n"
        "  2. Paper B — Jones et al. (2024)"
    )

def execute_summarize(args: SummarizeFindingsArgs) -> str:
    return (
        f"[summarize_findings] style={args.style}, max={args.max_sentences}\n"
        "  - Key finding 1\n  - Key finding 2\n  - Key finding 3"
    )

def run_tool(name, args):
    if name == "search_literature": return execute_search(args)
    if name == "summarize_findings": return execute_summarize(args)
    return f"[error] Unknown: {name}"

### Test End-to-End
Let's test the complete pipeline: user message → LLM with tools → Pydantic-validated args → tool execution with typed parameters.

In [ ]:
q = "Find 3 AlphaFold structure papers from 2022 in structural biology."
r = call_agent(q)
print("Tool:", r["tool"])
print("Args:", r["args"])
print()
if r["tool"]: print(run_tool(r["tool"], r["args"]))

<details>
<summary>Expected output (click to expand)</summary>

```
Tool: search_literature
Args: query='AlphaFold structure' max_results=3 year_from=2022 domain='structural biology'

[search_literature] 'AlphaFold structure' in structural biology from 2022+
  Returning up to 3 results (stub):
  1. Paper A — Smith et al. (2023)
  2. Paper B — Jones et al. (2024)
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **Design a Pydantic schema** for a tool in your scientific domain. What fields would it have? What types? What defaults?
2. **What happens when the LLM fills in wrong types?** (e.g., a string where an int is expected) How does Pydantic handle this?
3. **How would you add validation rules** beyond types? For example, ensuring `max_results` is between 1 and 100, or that a gene name matches a known format?

## Summary

Pydantic schemas give you:
- **Type safety**: `year_from: int | None`
- **Defaults**: `max_results=5` filled automatically
- **Documentation**: `description=` tells the LLM what each field means
- **Validation**: Pydantic raises clear errors on type mismatches

**Next**: Lab 3 — multi-turn memory.

---
*Agentic AI Science Playbook — Foundation Layer, Lab 2.*